In [38]:
import pandas as pd
import numpy as np
import biom

import os
from Bio import SeqIO

### getting metadata_v2.csv project id

In [4]:
def find_folders_with_metadata(root_path):
    """
    Find all folders containing metadata_v2.csv under the specified path.
    :param root_path: The root directory path to search.
    :return: A list of folder names that contain metadata_v2.csv.
    """
    matched_folders = []

    for folder_name in os.listdir(root_path):
        folder_path = os.path.join(root_path, folder_name)
        
        # 确保是文件夹
        if os.path.isdir(folder_path):
            metadata_path = os.path.join(folder_path, "metadata_v2.csv")
            if os.path.isfile(metadata_path):
                matched_folders.append(folder_name)

    return matched_folders

In [6]:
root_path = "../data/projects"
project_id = find_folders_with_metadata(root_path)

In [8]:
len(project_id)

334

### Getting DADA2.biom

In [74]:
# for i in project_id:
#     biom_table = pd.read_csv(f"/beegfs/dongbiao/oral/{i}/results/ASVs_counts.tsv", sep="\t", index_col=0)
#     table = biom.Table(
#         data=biom_table.values,                       
#         observation_ids=biom_table.index.tolist(),      
#         sample_ids=biom_table.columns.tolist()    
#     )
    
#     output_path = f"/beegfs/dongbiao/oral/{i}/results/ASVs_counts.biom"
#     with biom.util.biom_open(output_path, 'w') as f:
#         table.to_hdf5(f, generated_by="pandas_to_biom")

In [75]:
# for i in project_id:
#     seq_dict = {}
#     for record in SeqIO.parse(f"/beegfs/dongbiao/oral/{i}/results/ASVs.fa", "fasta"):
#         seq_dict[record.id] = str(record.seq)
#     biom_table = biom.load_table(f"/beegfs/dongbiao/oral/{i}/results/ASVs_counts.biom")
#     updated_table = biom_table.update_ids(seq_dict, axis='observation', inplace=False)
#     with biom.util.biom_open(f"/beegfs/dongbiao/oral/{i}/results/DADA2.biom", 'w') as f:
#         updated_table.to_hdf5(f, generated_by="fid_replacement")

### check the samples between DADA2.biom and metadata_v2.csv (sample_id)

In [54]:
results = []

for i in project_id: 
    
    biom_table = biom.load_table(f"/beegfs/dongbiao/oral/{i}/results/DADA2.biom")
    biom_sid = set(biom_table.ids(axis="sample"))
    
    metadata = pd.read_csv(f"../data/projects/{i}/metadata_v2.csv", dtype={"sample_id": str})
    metadata_sid = set(metadata.sample_id.values)
    
    matched       = biom_sid & metadata_sid          
    only_in_biom  = biom_sid - metadata_sid         
    only_in_meta  = metadata_sid - biom_sid          
    
    total_unique  = len(biom_sid | metadata_sid)    
    match_pct     = len(matched) / total_unique * 100 if total_unique > 0 else 0
    
    results.append({
        "project":            i,
        "biom_count":         len(biom_sid),
        "metadata_count":     len(metadata_sid),
        "matched_count":      len(matched),
        "match_pct":          round(match_pct, 2),
        "only_in_biom_count": len(only_in_biom),
        "only_in_meta_count": len(only_in_meta),
        "only_in_biom":       sorted(only_in_biom),  
        "only_in_meta":       sorted(only_in_meta),  
    })

In [57]:
results = pd.DataFrame(results)
results.sort_values("match_pct")

,project,biom_count,metadata_count,matched_count,match_pct,only_in_biom_count,only_in_meta_count,only_in_biom,only_in_meta
330,PRJEB1799,1061,848,842,78.91,219,6,"[ERR261870, ERR261879, ERR261885, ERR261889, E...","[ERR261988, ERR262251, ERR262311, ERR262377, E..."
250,PRJEB29801,47,59,47,79.66,0,12,[],"[ERR2930088, ERR2930089, ERR2930094, ERR293009..."
324,PRJNA316469,325,290,287,87.50,38,3,"[SRR3991820, SRR3991826, SRR3991835, SRR399184...","[SRR3991859, SRR3991956, SRR3992105]"
160,PRJEB11419,2116,2307,2116,91.72,0,191,[],"[ERR1073492, ERR1073494, ERR1073567, ERR107358..."
238,PRJNA362643,9269,8913,8876,95.38,393,37,"[SRR6385587, SRR6385588, SRR6385646, SRR638564...","[SRR6385583, SRR6385982, SRR6386012, SRR638611..."
...,...,...,...,...,...,...,...,...,...
112,PRJNA760654,166,166,166,100.00,0,0,[],[]
109,PRJEB40804,92,92,92,100.00,0,0,[],[]
108,PRJNA602902,119,119,119,100.00,0,0,[],[]
120,PRJNA649074,103,103,103,100.00,0,0,[],[]


In [53]:
### remove more samples from biom
# id_biom_sid_more = result_df.loc[(result_df['only_in_meta_count'] == 0) & (result_df['only_in_biom_count'] > 0)].project.values
# for i in id_biom_sid_more:
#     biom_table = biom.load_table(f"/beegfs/dongbiao/oral/{i}/results/DADA2.biom")
#     biom_sid = biom_table.ids(axis="sample")
    
#     metadata = pd.read_csv(f"../data/projects/{i}/metadata_v2.csv")
#     metadata_sid = metadata.sample_id.values
    
#     inter_sid = np.intersect1d(biom_sid, metadata_sid)
#     biom_table.filter(inter_sid, axis="sample")
#     biom_table.remove_empty()
#     with biom.util.biom_open(f"/beegfs/dongbiao/oral/{i}/results/DADA2.biom", 'w') as f:
#         biom_table.to_hdf5(f, generated_by="fid_replacement")

In [72]:
metadata_sid_more_project_id = results.loc[results.only_in_meta_count.values > 0].project.values
metadata_sid_more_project_id = metadata_sid_more_project_id[[i != "qiita_11404" for i in metadata_sid_more_project_id]]

In [73]:
metadata_sid_more_project_id

array(['PRJNA383868', 'PRJNA495719', 'PRJNA427101', 'PRJNA451534',
       'PRJEB6047', 'PRJNA380250', 'PRJEB24929', 'PRJNA595346',
       'PRJNA386665', 'PRJEB20890', 'PRJDB3518', 'PRJNA420339',
       'PRJEB27304', 'PRJEB5726', 'PRJNA647054', 'PRJNA853196',
       'PRJEB20889', 'PRJNA326441', 'PRJNA545251', 'PRJEB69530',
       'PRJEB20894', 'PRJEB19825', 'PRJNA417321', 'PRJEB39420',
       'PRJEB11419', 'PRJEB4953', 'PRJEB20893', 'PRJNA352060',
       'PRJEB19122', 'PRJEB11425', 'PRJNA362643', 'PRJNA395436',
       'PRJEB66628', 'PRJEB29801', 'PRJNA325500', 'PRJEB6518',
       'PRJNA503194', 'PRJNA321349', 'PRJEB36860', 'PRJNA414355',
       'PRJEB33119', 'PRJNA847970', 'PRJDB5440', 'PRJEB20891',
       'PRJEB20892', 'PRJEB38062', 'PRJNA316469', 'PRJEB1799',
       'PRJNA48333'], dtype=object)

In [77]:
sra_results = []

for i in metadata_sid_more_project_id:
 
    metadata = pd.read_csv(f"../data/projects/{i}/metadata_v2.csv", dtype={"sample_id": str})
    metadata_sid = set(metadata.sample_id.values)

    sra_path = f"/beegfs/dongbiao/oral/{i}/sra"
    if os.path.exists(sra_path):
        sra_folders = set(os.listdir(sra_path))
    else:
        sra_folders = set()
    
    matched          = sra_folders & metadata_sid
    only_in_sra      = sra_folders - metadata_sid   
    only_in_metadata = metadata_sid - sra_folders   
    
    total_unique = len(sra_folders | metadata_sid)
    match_pct    = len(matched) / total_unique * 100 if total_unique > 0 else 0
    
    sra_results.append({
        "project":              i,
        "metadata_count":       len(metadata_sid),
        "sra_folder_count":     len(sra_folders),
        "matched_count":        len(matched),
        "match_pct":            round(match_pct, 2),
        "only_in_sra_count":    len(only_in_sra),
        "only_in_meta_count":   len(only_in_metadata),
        "only_in_sra":          sorted(only_in_sra),
        "only_in_metadata":     sorted(only_in_metadata),  
    })

In [78]:
sra_df = pd.DataFrame(sra_results)

# Overview
print(f"Total projects analyzed: {len(sra_df)}")
print(f"Fully matched (100%):    {(sra_df['match_pct'] == 100).sum()}")
print(f"Projects with missing downloads: {(sra_df['only_in_meta_count'] > 0).sum()}")
print(f"Projects with extra downloads:   {(sra_df['only_in_sra_count'] > 0).sum()}")

sra_df

Total projects analyzed: 49
Fully matched (100%):    45
Projects with missing downloads: 4
Projects with extra downloads:   0


,project,metadata_count,sra_folder_count,matched_count,match_pct,only_in_sra_count,only_in_meta_count,only_in_sra,only_in_metadata
0,PRJNA383868,540,540,540,100.00,0,0,[],[]
1,PRJNA495719,90,90,90,100.00,0,0,[],[]
2,PRJNA427101,1319,1319,1319,100.00,0,0,[],[]
3,PRJNA451534,386,386,386,100.00,0,0,[],[]
4,PRJEB6047,72,72,72,100.00,0,0,[],[]
5,PRJNA380250,48,48,48,100.00,0,0,[],[]
6,PRJEB24929,231,231,231,100.00,0,0,[],[]
7,PRJNA595346,1834,1834,1834,100.00,0,0,[],[]
8,PRJNA386665,376,376,376,100.00,0,0,[],[]
9,PRJEB20890,397,397,397,100.00,0,0,[],[]


These 4 projects showed incomplete downloads, where certain samples present in the metadata could not be retrieved. This discrepancy is not due to data processing errors but rather download failures during the prefetch step. Specifically, some accession IDs could not be resolved within the SRA virtual file system, returning a 404 "name not found" error, as shown below:
2026-05-11T06:51:00 prefetch.3.0.0 err: name not found while resolving query within virtual file system module - failed to resolve accession 'ERR2930136' - no data ( 404 )
This indicates that the raw sequencing data for these accessions is either no longer publicly available, has been removed, or was never properly deposited in the SRA database. As a result, these samples are excluded from downstream analysis due to inaccessible source data rather than any quality-based filtering.